**YouTube API Credentials Setup**

In [ ]:
import time
from googleapiclient.discovery import build
import googleapiclient.discovery
api_key = ''
youtube = build('youtube', 'v3', developerKey=api_key)
api_service_name = "youtube"
api_version = "v3"

**Configuración de scispaCy**

In [ ]:
import scispacy
import spacy
import en_core_sci_sm
from spacy import displacy
import pandas as pd

**Function to search videos by deciding the number of videos you want to search**

In [3]:
youtube = build(api_service_name, api_version, developerKey=api_key)

def buscar_videos(query, max_results):
    videos = []
    next_page_token = None

    while len(videos) < max_results:
        request = youtube.search().list(
            part="id",
            maxResults=min(max_results - len(videos), 50),  # Limitar a 50 por solicitud
            q=query,
            videoCaption="closedCaption",
            type="video",
            order="viewCount",
            relevanceLanguage="es",
            pageToken=next_page_token
        )
        response = request.execute()

        for item in response.get("items", []):
            videos.append(item)

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

    return videos

**SEARCH TERM LISTS**

- **Terms for the psuedoscientific corpus:** Terapia energética, Reiki, Naturopatía, Acupuntura, Magnetoterapia, Flores de Bach.

- **Terms for the scientific corpus:** Tratamiento cáncer, Procedimiento endóscopico, Terapia Cognitivo Conductual, Neurorrehabilitación, Fisioterapia deportiva.

**CREATION OF VARIABLES THAT WILL BE USED TO DO THE SEARCH**

In [4]:
# psuedoscientific
pseudo_1 = "Terapia energética"
pseudo_2 = "Reiki"
pseudo_3 = "Naturopatía"
pseudo_4 = "Acupuntura"
pseudo_5 = "Magnetoterapia"
pseudo_6 = "Flores de Bach"

# scientific
scientific_1 = "Tratamiento cáncer"
scientific_2 = "Procedimiento endóscopico"
scientific_3 = "Terapia Cognitivo Conductual"
scientific_4 = "Neurorrehabilitación"
scientific_5 = "Fisioterapia deportiva"
scientific_6 = "Rehabilitación cardíaca"

**SEARCH WITH A CHOSEN SPECIFIC TERM**

It's necessary to run the code with each of the terms. A CSV file will be obtained for each search. To analyze them, it's necessary to combine them.

In [5]:
search_term = scientific_5
videos_iteracion =  buscar_videos(search_term, 100)
videos_iteracion = [item["id"]["videoId"] for item in videos_iteracion if "videoId" in item["id"]]
video_ids = videos_iteracion
video_ids

['YtgoxfG7uX8',
 '_9ElpXUjlR4',
 'Y0ccTBkTyfg',
 'utkdcx7Nwvk',
 'iVw3xIEAOx8',
 'VLPjNVBggTo',
 '57cGclIpSBA',
 'VyVoEKUsIo0',
 'AqKQjHJ8HN8',
 'nmnAf0zDLGE',
 '1eM1FSAS4CE',
 '_7-oAh4G0Ro',
 'HrYB128P94M',
 'r5S_T9ITiNw',
 'WsPt_cSX6dI',
 'HiywqhjAX_Q',
 'aU3LVtLLhww',
 '9akRmwI2X5U',
 'BXUqpgHz_so',
 'r-gEdBHB_V8',
 'dEF3eTAx_DE',
 'eDbc_JcIy-0',
 'itRE0LKft60',
 '6uFk9oeF1uM',
 'SwbvH2bIWbo',
 'xSkSgsbD72Q',
 'AAQxkPPB5U0',
 '7pnSOFEwwRk',
 'Wg9itsXbUFE',
 'D78luEXTrdI',
 'cTlZMI1SWow',
 'RAIufsQbN6I',
 'y2cpKLmagWw',
 'xapW1RS5JtA',
 'xbF81mVcmUU',
 'sinyV_4vHig',
 '32f-ewLsxx8',
 'r-ITuNPzxVU',
 'QL5qEwNz3bE']

**Setting up an LLM to categorize the videos**

In [6]:
import requests
import pathlib
import textwrap
import google.generativeai as genai
from IPython.display import display
from IPython.display import Markdown

def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

import os
GOOGLE_API_KEY=os.getenv('')
genai.configure(api_key=GOOGLE_API_KEY)
model = genai.GenerativeModel('gemini-2.0-flash-lite')
genai.configure(api_key='')

**Creating a dictionary with the information from each video**

In [7]:
# Extracting title, description and tags of each video
basic_info_dict = {}
for video_id in video_ids:
    try:
        request = youtube.videos().list(
            part="snippet",
            id=video_id
        )
        response = request.execute()

        if response and "items" in response and len(response["items"]) > 0:
            snippet = response["items"][0]["snippet"]
            title = snippet.get("title")
            description = snippet.get("description")
            tags = snippet.get("tags") if "tags" in snippet else []

            video_info = {
                "title": title,
                "description": description,
                "tags": tags
            }
            basic_info_dict[video_id] = video_info
    except: # If there is any problem with the video, we pass to the next one
        pass

In [11]:
len(english_video_info.items())

39

**Video classification**

**Prompt**

In [ ]:
# Base prompt
prompt_base = """
A continuación recibirás la información básica de un vídeo de youtube que deberás usar para clasificar el vídeo en una de las siguientes categorías:

Científico: El vídeo presenta información basada en evidencia empírica, métodos científicos y consenso científico.
Pseudocientífico: El vídeo presenta información que se disfraza de ciencia, pero carece de evidencia empírica, métodos científicos rigurosos o consenso científico.
Irrelevante: El contenido del vídeo no está relacionado con ciencia o pseudociencia, o es difícil de clasificar debido a su ambigüedad o falta de claridad. Si la información provista es muy breve y genérica, clasifícalo como irrelevante.

Devuelve únicamente la etiqueta de la categoría. Escribe exclusivamente la palabra, no quiero que haya nada más tras ella. Ni siquiera un salto de linea o un punto.

A continuación se provee el título del vídeo, la descripción del mismo, y sus etiquetas. En ocasiones alguno de estos campos puede estar vacío. En este caso haz la evaluación con la información que tienes disponible. Si no hay información suficiente para determinar con precision la categoría, simplemente clasificalo como "Irrelevante".

Título: {}

Descripción del vídeo: {}

Etiquetas del vídeo: {}
"""

**Creation of a dictionary that contains the original information within this prompt**

In [ ]:
# Creating a dictionary with a custom prompt for each video
dict_prompt_info = {}

for video_id, transcripcion in basic_info_dict.items():
    prompt_con_transcripcion = prompt_base.format(transcripcion['title'], transcripcion['description'], transcripcion['tags'])
    dict_prompt_info[video_id] = prompt_con_transcripcion

**Categorizing videos using Google Gemini API calls**

In [ ]:
import time

dict_respuestas_llm = {}
consulta_contador = 0
for video_id, prompt_con_transcripcion in dict_prompt_info.items():
    response = model.generate_content(prompt_con_transcripcion)
    dict_respuestas_llm[video_id] = response.text
    consulta_contador += 1
    if consulta_contador % 14 == 0:
        print("Pausando por un minuto...")
        time.sleep(60)
        print("Continuando...")

dict_respuestas_llm

**We include the result of this analysis in the previous dataframe to have a unified dataframe**

In [ ]:
classification_df = pd.DataFrame.from_dict(dict_respuestas_llm, orient='index', columns=['Content_Classification'])
final_df = results_df.join(classification_df, how='left')
final_df

**Export results to a CSV file**

In [ ]:
import csv
# Name of csv
archivo_csv = "Busqueda_" + search_term + ".csv"
final_df.to_csv(archivo_csv, index=True, encoding='utf-8')